In [1]:
import tensorflow as tf
import time
import numpy as np
import os
import copy
import pickle
import argparse
import utilityarm as utility
import pandas as pd
from sklearn.metrics import *

In [2]:
import tensorflow.compat.v1 as tf

tf.disable_v2_behavior() 

class FairAdvBPR:

    def __init__(self, sess, dict_args, train_df, test_df, user_type, type_error_weight, key_type, user_type_list, item_type_count):
       
        self.dataname = dict_args['dataname']

        self.key_type = key_type
        self.user_type_list = user_type_list
        self.item_type_count = item_type_count
        self.layers = dict_args['layers']
        self.sess = sess
        
        self.num_cols = len(train_df['item_id'].unique())
        self.num_rows = len(train_df['user_id'].unique())

        self.hidden_neuron = dict_args['hidden_neuron']
        self.neg = dict_args['neg']
        self.batch_size = dict_args['batch_size']

        self.train_df = train_df
        self.vali_df = test_df
        self.num_train = len(self.train_df)
        self.num_vali = len(self.vali_df)

        self.train_epoch = dict_args['train_epoch']
        self.train_epoch_a = dict_args['train_epoch_a']

        self.lr_r = dict_args['lr_r'] # learning rate
        self.lr_a = dict_args['lr_a'] # learning rate
        self.alpha = dict_args['alpha'] # learning rate
        self.optimizer_method = dict_args['optimizer_method']
        self.display_step = dict_args['display_step']
        
        self.type_error_weight = type_error_weight
        self.num_type = dict_args['num_type']
        
        self.user_type = user_type
        self.type_count_list = []
        for k in range(self.num_type):
            self.type_count_list.append(np.sum(user_type[:,k]))

        
        self.reg = dict_args['reg'] # regularization term trade-off
        self.reg_s = dict_args['reg_s']

        print('**********fairAdvBPR**********')
        #print(self.args)
        self._prepare_model()

    def loadmodel(self, saver, checkpoint_dir):
        ckpt = tf.train.get_checkpoint_state(checkpoint_dir)
        if ckpt and ckpt.model_checkpoint_path:
            ckpt_name = os.path.basename(ckpt.model_checkpoint_path)
            saver.restore(self.sess, os.path.join(checkpoint_dir, ckpt_name))
            return True
        else:
            return False
        
    def run(self):
        init = tf.global_variables_initializer()
        self.sess.run(init)

        saver = tf.train.Saver([self.P, self.Q])
        self.loadmodel(saver, "./"+self.dataname+"/BPR_check_points")

        for epoch_itr in range(1, self.train_epoch + 1 + self.train_epoch_a):
            self.train_model(epoch_itr)
            if epoch_itr % self.display_step == 0:
                self.test_model(epoch_itr)
        return self.make_records()

    def _prepare_model(self):
        with tf.name_scope("input_data"):
            self.user_input = tf.placeholder(tf.int32, shape=[None, 1], name="user_input")
            self.item_input_pos = tf.placeholder(tf.int32, shape=[None, 1], name="item_input_pos")
            self.item_input_neg = tf.placeholder(tf.int32, shape=[None, 1], name="item_input_neg")

            self.input_user_type = tf.placeholder(dtype=tf.float32, shape=[None, self.num_type]
                                                   , name="input_user_type")
            self.input_user_error_weight = tf.placeholder(dtype=tf.float32, shape=[None, 1]
                                                          , name="input_user_error_weight")

        with tf.variable_scope("BPR", reuse=tf.AUTO_REUSE):
            self.P = tf.get_variable(name="P",
                                     initializer=tf.truncated_normal(shape=[self.num_rows, self.hidden_neuron], mean=0,
                                                                     stddev=0.03), dtype=tf.float32)
            self.Q = tf.get_variable(name="Q",
                                     initializer=tf.truncated_normal(shape=[self.num_cols+1, self.hidden_neuron], mean=0,
                                                                     stddev=0.03), dtype=tf.float32)
        para_r = tf.get_collection(tf.GraphKeys.GLOBAL_VARIABLES, scope="BPR")

        with tf.variable_scope("Adversarial", reuse=tf.AUTO_REUSE):
            num_layer = len(self.layers)
            adv_W = []
            adv_b = []
            for l in range(num_layer):
                if l == 0:
                    in_shape = 21
                else:
                    in_shape = self.layers[l - 1]
                adv_W.append(tf.get_variable(name="adv_W" + str(l),
                                             initializer=tf.truncated_normal(shape=[in_shape, self.layers[l]],
                                                                             mean=0, stddev=0.03), dtype=tf.float32))
                adv_b.append(tf.get_variable(name="adv_b" + str(l),
                                             initializer=tf.truncated_normal(shape=[1, self.layers[l]],
                                                                             mean=0, stddev=0.03), dtype=tf.float32))
            adv_W_out = tf.get_variable(name="adv_W_out",
                                        initializer=tf.truncated_normal(shape=[self.layers[-1], self.num_type],
                                                                        mean=0, stddev=0.03), dtype=tf.float32)

            adv_b_out = tf.get_variable(name="adv_b_out",
                                        initializer=tf.truncated_normal(shape=[1, self.num_type],
                                                                        mean=0, stddev=0.03), dtype=tf.float32)
        para_a = tf.get_collection(tf.GraphKeys.GLOBAL_VARIABLES, scope="Adversarial")

        p = tf.reduce_sum(tf.nn.embedding_lookup(self.P, self.user_input), 1)
        q_neg = tf.reduce_sum(tf.nn.embedding_lookup(self.Q, self.item_input_neg), 1)
        q_pos = tf.reduce_sum(tf.nn.embedding_lookup(self.Q, self.item_input_pos), 1)

        predict_pos = tf.reduce_sum(p * q_pos, 1)
        predict_neg = tf.reduce_sum(p * q_neg, 1)

        r_cost1 = tf.reduce_sum(tf.nn.softplus(-(predict_pos - predict_neg)))
        r_cost2 = self.reg * 0.5 * (self.l2_norm(self.P) + self.l2_norm(self.Q))  # regularization term
        pred = tf.matmul(self.P, tf.transpose(self.Q))
        self.s_mean = tf.reduce_mean(pred, axis=1)
        self.s_std = tf.keras.backend.std(pred, axis=1)
        self.s_cost = tf.reduce_sum(tf.square(self.s_mean) + tf.square(self.s_std) - 2 * tf.log(self.s_std) - 1)
        self.r_cost = r_cost1 + r_cost2 + self.reg_s * 0.5 * self.s_cost
        
        print('shape q pos',q_pos.shape)
        print('shape p',p.shape)
        print('shape predict pos ', predict_pos.shape)
        
        adv_last = tf.reshape(predict_pos, [tf.shape(self.input_user_type)[0], 1])
        print('shape adv_last ', adv_last.shape)
        adv_last = tf.concat([adv_last, q_pos], 1)
        
        for l in range(num_layer):
            adv = tf.nn.relu(tf.matmul(adv_last, adv_W[l]) + adv_b[l])
            adv_last = adv
        self.adv_output = tf.nn.sigmoid(tf.matmul(adv_last, adv_W_out) + adv_b_out)
        self.a_cost = tf.reduce_sum(tf.square(self.adv_output - self.input_user_type) * self.input_user_error_weight)

        self.all_cost = self.r_cost - self.alpha * self.a_cost  # the loss function

        with tf.variable_scope("Optimizer", reuse=tf.AUTO_REUSE):
            self.r_optimizer = tf.train.AdamOptimizer(learning_rate=self.lr_r).minimize(self.r_cost, var_list=para_r)
            self.a_optimizer = tf.train.AdamOptimizer(learning_rate=self.lr_a).minimize(self.a_cost, var_list=para_a)
            self.all_optimizer = tf.train.AdamOptimizer(learning_rate=self.lr_r).minimize(self.all_cost, var_list=para_r)


    def train_model(self, itr):
        NS_start_time = time.time() * 1000.0
        epoch_r_cost = 0.0
        epoch_s_cost = 0.0
        epoch_s_mean = 0.0
        epoch_s_std = 0.0
        epoch_a_cost = 0.0
        num_sample, user_list, item_pos_list, item_neg_list = utility.negative_sample(self.train_df, self.num_rows,
                                                                                      self.num_cols, self.neg)
        NS_end_time = time.time() * 1000.0

        start_time = time.time() * 1000.0
        num_batch = int(num_sample / float(self.batch_size)) + 1
        random_idx = np.random.permutation(num_sample)
        for i in range(num_batch):
            # get the indices of the current batch
            if i == num_batch - 1:
                batch_idx = random_idx[i * self.batch_size:]
            elif i < num_batch - 1:
                batch_idx = random_idx[(i * self.batch_size):((i + 1) * self.batch_size)]

            if itr > self.train_epoch:
                random_idx_a = np.random.permutation(num_sample)
                print("boucle adversarial debut-- num batch ",i)
                for j in range(num_batch):
                    if j == num_batch - 1:
                        batch_idx_a = random_idx_a[j * self.batch_size:]
                    elif j < num_batch - 1:
                        batch_idx_a = random_idx_a[(j * self.batch_size):((j + 1) * self.batch_size)]
                    user_idx_list = ((user_list[batch_idx_a, :]).reshape((len(batch_idx_a)))).tolist()
                    _, tmp_a_cost = self.sess.run(  # do the optimization by the minibatch
                        [self.a_optimizer, self.a_cost],
                        feed_dict={self.user_input: user_list[batch_idx_a, :],
                                   self.item_input_pos: item_pos_list[batch_idx_a, :],
                                   self.item_input_neg: item_neg_list[batch_idx_a, :],
                                   self.input_user_type: self.user_type[user_idx_list,:],
                                   self.input_user_error_weight: self.type_error_weight[user_idx_list,:]})
                    epoch_a_cost += tmp_a_cost

                user_idx_list = ((user_list[batch_idx, :]).reshape((len(batch_idx)))).tolist()
                _, tmp_r_cost, tmp_s_cost, tmp_s_mean, tmp_s_std = self.sess.run(  # do the optimization by the minibatch
                    [self.all_optimizer, self.all_cost, self.s_cost, self.s_mean, self.s_std],
                    feed_dict={self.user_input: user_list[batch_idx, :],
                               self.item_input_pos: item_pos_list[batch_idx, :],
                               self.item_input_neg: item_neg_list[batch_idx, :],
                               self.input_user_type: self.user_type[user_idx_list, :],
                               self.input_user_error_weight: self.type_error_weight[user_idx_list, :]})
                epoch_r_cost += tmp_r_cost
                epoch_s_mean += np.mean(tmp_s_mean)
                epoch_s_std += np.mean(tmp_s_std)
                epoch_s_cost += tmp_s_cost
                print("boucle adversarial fin")
            else:
                user_idx_list = ((user_list[batch_idx, :]).reshape((len(batch_idx)))).tolist()
                _, tmp_r_cost, tmp_s_cost, tmp_s_mean, tmp_s_std = self.sess.run(  # do the optimization by the minibatch
                    [self.r_optimizer, self.r_cost, self.s_cost, self.s_mean, self.s_std],
                    feed_dict={self.user_input: user_list[batch_idx, :],
                               self.item_input_pos: item_pos_list[batch_idx, :],
                               self.item_input_neg: item_neg_list[batch_idx, :],
                               self.input_user_type: self.user_type[user_idx_list, :],
                               self.input_user_error_weight: self.type_error_weight[user_idx_list, :]})
                epoch_r_cost += tmp_r_cost
                epoch_s_mean += np.mean(tmp_s_mean)
                epoch_s_std += np.mean(tmp_s_std)
                epoch_s_cost += tmp_s_cost
        epoch_a_cost /= num_batch
        if itr % self.display_step == 0:
            print ("Training //", "Epoch %d //" % itr, " Total r_cost = %.5f" % epoch_r_cost,
                   " Total s_cost = %.5f" % epoch_s_cost,
                   " Total s_mean = %.5f" % epoch_s_mean,
                   " Total s_std = %.5f" % epoch_s_std,
                   " Total a_cost = %.5f" % epoch_a_cost,
                   "Training time : %d ms" % (time.time() * 1000.0 - start_time),
                   "negative Sampling time : %d ms" % (NS_end_time - NS_start_time),
                   "negative samples : %d" % (num_sample))
       
    def test_model(self, itr):  # calculate the cost and rmse of testing set in each epoch
        if itr % self.display_step == 0:
            start_time = time.time() * 1000.0
            P, Q = self.sess.run([self.P, self.Q])
            Rec = np.matmul(P, Q.T)

            [precision, recall, f_score, NDCG] = utility.test_model_all(Rec, self.vali_df, self.train_df)
#             utility.ranking_analysis(Rec, self.vali_df, self.train_df, self.key_genre, self.item_genre_list,
#                                      self.user_genre_count)
            utility.test_model_per_user_type(Rec, self.vali_df, self.train_df, self.user_type_list, self.key_type)
            auc = utility.auc_per_user(Rec, self.vali_df, self.train_df)
            print("AUC global is: ", auc)
            
            filename = './IembfairAdvBPR_results/epoch'+ str(itr) +'_Rec_' + self.dataname + '_iembfairAdvBPR.npy'
            os.makedirs(os.path.dirname(filename), exist_ok=True)           
            with open(filename, "wb") as f:
                np.save(f, Rec)
            

    def make_records(self):  # record all the results' details into files
        P, Q = self.sess.run([self.P, self.Q])
        Rec = np.matmul(P, Q.T)

        [precision, recall, f_score, NDCG] = utility.test_model_all(Rec, self.vali_df, self.train_df)
        return precision, recall, f_score, NDCG, Rec

#     def test_model(self, itr):  # calculate the cost and rmse of testing set in each epoch
#         if itr % self.display_step == 0:
#             start_time = time.time() * 1000.0
#             P, Q = self.sess.run([self.P, self.Q])
#             Rec = np.matmul(P, Q.T)

#             [precision, recall, f_score, NDCG] = utility.test_model_all(Rec, self.vali_df, self.train_df)
# #             utility.ranking_analysis(Rec, self.vali_df, self.train_df, self.key_type, self.user_type_list,
# #                                      self.item_type_count)
#             utility.test_model_per_user_type(Rec, self.vali_df, self.train_df, self.user_type_list, self.key_type)
#             auc = utility.auc_per_user(Rec, self.vali_df, self.train_df)
#             print("AUC global is: ", auc)
#             print (
#                 "Testing //", "Epoch %d //" % itr,
#                 "Testing time : %d ms" % (time.time() * 1000.0 - start_time))
#             print("=" * 200)


    @staticmethod
    def l2_norm(tensor):
        return tf.reduce_sum(tf.square(tensor))


Instructions for updating:
non-resource variables are not supported in the long term


In [3]:

#optimizer_method = ['Adam', 'Adadelta', 'Adagrad', 'RMSProp', 'GradientDescent','Momentum'], default='Adam')


train_epoch = 1
train_epoch_a = 20 #default 20
display_step = 1
lr_r = 0.01
lr_a = 0.005
reg = 0.1
reg_s = 30
alpha = 1000
optimizer_method = 'Adam'
hidden_neuron = 20
n = 1
neg = 5
batch_size = 256
layers = [50, 50, 50, 50]
dataname = 'ml1m-6'

In [4]:
dict_args =  {"train_epoch": train_epoch,
              "train_epoch_a": train_epoch_a,
            "display_step":display_step,
            "lr_r":lr_r,
            "lr_a":lr_a,
            "reg":reg,
            "reg_s":reg_s,
            "alpha":alpha,
            "optimizer_method":optimizer_method,
            "hidden_neuron":hidden_neuron,
            "n":n,
            "neg":neg,
            "batch_size":batch_size,
            "layers":layers,
            "dataname":dataname}
dict_args

{'train_epoch': 1,
 'train_epoch_a': 1,
 'display_step': 1,
 'lr_r': 0.01,
 'lr_a': 0.005,
 'reg': 0.1,
 'reg_s': 30,
 'alpha': 1000,
 'optimizer_method': 'Adam',
 'hidden_neuron': 20,
 'n': 1,
 'neg': 5,
 'batch_size': 1024,
 'layers': [50, 50, 50, 50],
 'dataname': 'ml1m-6'}

In [5]:
with open('./training_df.pkl', 'rb') as f:
    train_df = pickle.load(f,encoding='latin1')

# with open('./' + dataname + '/valiing_df.pkl', 'rb') as f:
#     vali_df = pickle.load(f,encoding='latin1')  # for validation
    
with open('./testing_df.pkl', 'rb') as f:
    test_df = pickle.load(f,encoding='latin1')  # for validation
# vali_df = pickle.load(open('./' + dataname + '/testing_df.pkl'))  # for testing

with open('./key_type.pkl', 'rb') as f:
    key_type = pickle.load(f,encoding='latin1')
    
with open('./user_idd_type_list.pkl', 'rb') as f:
    user_idd_type_list = pickle.load(f,encoding='latin1')
    
with open('./type_user_vector.pkl', 'rb') as f:
    type_user_vector = pickle.load(f,encoding='latin1')

with open('./type_count.pkl', 'rb') as f:
    type_count = pickle.load(f,encoding='latin1')
    
with open('./item_type_count.pkl', 'rb') as f:
    item_type_count = pickle.load(f,encoding='latin1')

In [6]:
train_df.head(20)

,user_id,item_id,rating
0,0,0,5
1,0,2,3
2,0,3,4
3,0,4,5
4,0,5,3
5,0,7,5
6,0,9,4
7,0,10,5
8,0,11,4
9,0,12,4


In [7]:
train_df.shape

(802322, 3)

In [8]:
test_df.head(20)

,user_id,item_id,rating
0,0,1,3
1,0,6,5
2,0,8,4
3,0,13,4
4,0,17,4
5,0,28,4
6,0,39,5
7,0,42,4
8,0,43,4
9,0,50,4


In [9]:
test_df.shape

(197595, 3)

In [10]:
print(len(user_idd_type_list))

6040


In [11]:
user_idd_type_list

array([['F'],
       ['M'],
       ['M'],
       ...,
       ['F'],
       ['F'],
       ['M']], dtype='<U1')

In [12]:
print(type_user_vector['F'].shape)

(1, 6040)


In [13]:
type_user_vector

{'M': array([[0., 1., 1., ..., 0., 0., 1.]]),
 'F': array([[1., 0., 0., ..., 1., 1., 0.]])}

In [14]:
len(item_type_count)

3503

In [15]:
item_type_count

[{'F': 1353, 'M': 3293},
 {'F': 1578, 'M': 4041},
 {'F': 1464, 'M': 4053},
 {'F': 1362, 'M': 3596},
 {'F': 1335, 'M': 3335},
 {'F': 1192, 'M': 2962},
 {'F': 1598, 'M': 3863},
 {'F': 1434, 'M': 3530},
 {'F': 1521, 'M': 3935},
 {'F': 1301, 'M': 3378},
 {'F': 1400, 'M': 3779},
 {'F': 1630, 'M': 4253},
 {'F': 1609, 'M': 4115},
 {'F': 1407, 'M': 3442},
 {'F': 1482, 'M': 3847},
 {'F': 1381, 'M': 3271},
 {'F': 1605, 'M': 4086},
 {'F': 1549, 'M': 4014},
 {'F': 1540, 'M': 3955},
 {'F': 1393, 'M': 3443},
 {'F': 1450, 'M': 3677},
 {'F': 1626, 'M': 4064},
 {'F': 1197, 'M': 2764},
 {'F': 1218, 'M': 2965},
 {'F': 1606, 'M': 4152},
 {'F': 1586, 'M': 4158},
 {'F': 1222, 'M': 2972},
 {'F': 1313, 'M': 3467},
 {'F': 1680, 'M': 4303},
 {'F': 1567, 'M': 3943},
 {'F': 1563, 'M': 3950},
 {'F': 1570, 'M': 4119},
 {'F': 1595, 'M': 4080},
 {'F': 1356, 'M': 3576},
 {'F': 1581, 'M': 4063},
 {'F': 1589, 'M': 4135},
 {'F': 1675, 'M': 4260},
 {'F': 1537, 'M': 4044},
 {'F': 1186, 'M': 2893},
 {'F': 1443, 'M': 3564},


In [16]:
print(type_count)

{'F': 1709, 'M': 4331}


In [17]:
num_item = len(train_df['item_id'].unique())
num_user = len(train_df['user_id'].unique())
num_type = len(key_type)
print('items number : ',num_item)
print('users number : ',num_user)
print('user types : ',key_type)


items number :  3503
users number :  6040
user types :  ['M', 'F']


In [18]:
dict_args["num_type"] = len(key_type)

In [19]:
user_type_list = [] #preprocessing to be sure that user types are really the right ones armielle 
for u in range(num_user):
    gl = user_idd_type_list[u]
    tmp = []
    for g in gl:
        if g in key_type:
            tmp.append(g)
    user_type_list.append(tmp)

print(len(user_type_list))

6040


In [20]:
# genreate user_type matrix
user_type = np.zeros((num_user, num_type))
for u in range(num_user):
    gl = user_type_list[u]
    for k in range(num_type):
        if key_type[k] in gl:
            user_type[u, k] = 1.0

In [21]:
len(user_type_list)

6040

In [22]:
print('*' * 50)
print('number of positive feedback: ' + str(len(train_df)))
print('estimated number of training samples: ' + str(neg * len(train_df)))
print('*' * 50)

**************************************************
number of positive feedback: 802322
estimated number of training samples: 4011610
**************************************************


In [23]:
type_count_mean_reciprocal = []
for k in key_type:
    type_count_mean_reciprocal.append(1.0 / type_count[k])
type_count_mean_reciprocal = (np.array(type_count_mean_reciprocal)).reshape((num_type, 1))
type_error_weight = np.dot(user_type, type_count_mean_reciprocal)


In [24]:
# generate user_type matrix
type_user_indicator = np.zeros((num_type, num_user))

for k in range(num_type):
    type_user_indicator[k,:] = type_user_vector[key_type[k]]


In [25]:
precision = np.zeros(4)
recall = np.zeros(4)
f1 = np.zeros(4)
ndcg = np.zeros(4)
RSP = np.zeros(4)
REO = np.zeros(4)

precision 

array([0., 0., 0., 0.])

In [26]:
len(user_type_list)

6040

In [27]:
tf.compat.v1.disable_eager_execution()

for i in range(n):
    with tf.compat.v1.Session() as sess:
        fairadvbpr = FairAdvBPR(sess, dict_args, train_df, test_df, user_type, type_error_weight, key_type, user_type_list, item_type_count)
        [prec_one, rec_one, f_one, ndcg_one, Rec] = fairadvbpr.run()
        #[RSP_one, REO_one] = utility.ranking_analysis(Rec, vali_df, train_df, key_genre, item_genre_list, user_genre_count)
#         precision += prec_one
#         recall += rec_one
#         f1 += f_one
#         ndcg += ndcg_one
#         RSP += RSP_one
#         REO += REO_one

**********fairAdvBPR**********
shape q pos (?, 20)
shape p (?, 20)
shape predict pos  (?,)
shape adv_last  (?, 1)
INFO:tensorflow:Restoring parameters from ./ml1m-6/BPR_check_points\check_point.ckpt-1
Training // Epoch 1 //  Total r_cost = 5383379.62384  Total s_cost = 127782.06882  Total s_mean = -1.75820  Total s_std = 3906.86892  Total a_cost = 0.00000 Training time : 751472 ms negative Sampling time : 174120 ms negative samples : 4011610
precision_1	[0.2561258],	||	 precision_5	[0.2046689],	||	 precision_10	[0.1792550],	||	 precision_15	[0.1634768]
recall_1   	[0.0098782],	||	 recall_5   	[0.0370207],	||	 recall_10   	[0.0640690],	||	 recall_15   	[0.0860630]
f_measure_1	[0.0190227],	||	 f_measure_5	[0.0627002],	||	 f_measure_10	[0.0943983],	||	 f_measure_15	[0.1127620]
ndcg_1     	[0.2561258],	||	 ndcg_5     	[0.2146712],	||	 ndcg_10     	[0.1981298],	||	 ndcg_15     	[0.1902918]
Metrics for user type	 M
precision_1	[0.2897714],	||	 precision_5	[0.2295082],	||	 precision_10	[0.199

boucle adversarial fin
boucle adversarial debut-- num batch  103
boucle adversarial fin
boucle adversarial debut-- num batch  104
boucle adversarial fin
boucle adversarial debut-- num batch  105
boucle adversarial fin
boucle adversarial debut-- num batch  106
boucle adversarial fin
boucle adversarial debut-- num batch  107
boucle adversarial fin
boucle adversarial debut-- num batch  108
boucle adversarial fin
boucle adversarial debut-- num batch  109
boucle adversarial fin
boucle adversarial debut-- num batch  110
boucle adversarial fin
boucle adversarial debut-- num batch  111
boucle adversarial fin
boucle adversarial debut-- num batch  112
boucle adversarial fin
boucle adversarial debut-- num batch  113
boucle adversarial fin
boucle adversarial debut-- num batch  114
boucle adversarial fin
boucle adversarial debut-- num batch  115
boucle adversarial fin
boucle adversarial debut-- num batch  116
boucle adversarial fin
boucle adversarial debut-- num batch  117
boucle adversarial fin
bo

boucle adversarial debut-- num batch  229
boucle adversarial fin
boucle adversarial debut-- num batch  230
boucle adversarial fin
boucle adversarial debut-- num batch  231
boucle adversarial fin
boucle adversarial debut-- num batch  232
boucle adversarial fin
boucle adversarial debut-- num batch  233
boucle adversarial fin
boucle adversarial debut-- num batch  234
boucle adversarial fin
boucle adversarial debut-- num batch  235
boucle adversarial fin
boucle adversarial debut-- num batch  236
boucle adversarial fin
boucle adversarial debut-- num batch  237
boucle adversarial fin
boucle adversarial debut-- num batch  238
boucle adversarial fin
boucle adversarial debut-- num batch  239
boucle adversarial fin
boucle adversarial debut-- num batch  240
boucle adversarial fin
boucle adversarial debut-- num batch  241
boucle adversarial fin
boucle adversarial debut-- num batch  242
boucle adversarial fin
boucle adversarial debut-- num batch  243
boucle adversarial fin
boucle adversarial debut-

boucle adversarial fin
boucle adversarial debut-- num batch  356
boucle adversarial fin
boucle adversarial debut-- num batch  357
boucle adversarial fin
boucle adversarial debut-- num batch  358
boucle adversarial fin
boucle adversarial debut-- num batch  359
boucle adversarial fin
boucle adversarial debut-- num batch  360
boucle adversarial fin
boucle adversarial debut-- num batch  361
boucle adversarial fin
boucle adversarial debut-- num batch  362
boucle adversarial fin
boucle adversarial debut-- num batch  363
boucle adversarial fin
boucle adversarial debut-- num batch  364
boucle adversarial fin
boucle adversarial debut-- num batch  365
boucle adversarial fin
boucle adversarial debut-- num batch  366
boucle adversarial fin
boucle adversarial debut-- num batch  367
boucle adversarial fin
boucle adversarial debut-- num batch  368
boucle adversarial fin
boucle adversarial debut-- num batch  369
boucle adversarial fin
boucle adversarial debut-- num batch  370
boucle adversarial fin
bo

boucle adversarial fin
boucle adversarial debut-- num batch  483
boucle adversarial fin
boucle adversarial debut-- num batch  484
boucle adversarial fin
boucle adversarial debut-- num batch  485
boucle adversarial fin
boucle adversarial debut-- num batch  486
boucle adversarial fin
boucle adversarial debut-- num batch  487
boucle adversarial fin
boucle adversarial debut-- num batch  488
boucle adversarial fin
boucle adversarial debut-- num batch  489
boucle adversarial fin
boucle adversarial debut-- num batch  490
boucle adversarial fin
boucle adversarial debut-- num batch  491
boucle adversarial fin
boucle adversarial debut-- num batch  492
boucle adversarial fin
boucle adversarial debut-- num batch  493
boucle adversarial fin
boucle adversarial debut-- num batch  494
boucle adversarial fin
boucle adversarial debut-- num batch  495
boucle adversarial fin
boucle adversarial debut-- num batch  496
boucle adversarial fin
boucle adversarial debut-- num batch  497
boucle adversarial fin
bo

...

boucle adversarial fin
boucle adversarial debut-- num batch  3362
boucle adversarial fin
boucle adversarial debut-- num batch  3363
boucle adversarial fin
boucle adversarial debut-- num batch  3364
boucle adversarial fin
boucle adversarial debut-- num batch  3365
boucle adversarial fin
boucle adversarial debut-- num batch  3366
boucle adversarial fin
boucle adversarial debut-- num batch  3367
boucle adversarial fin
boucle adversarial debut-- num batch  3368
boucle adversarial fin
boucle adversarial debut-- num batch  3369
boucle adversarial fin
boucle adversarial debut-- num batch  3370
boucle adversarial fin
boucle adversarial debut-- num batch  3371
boucle adversarial fin
boucle adversarial debut-- num batch  3372
boucle adversarial fin
boucle adversarial debut-- num batch  3373
boucle adversarial fin
boucle adversarial debut-- num batch  3374
boucle adversarial fin
boucle adversarial debut-- num batch  3375
boucle adversarial fin
boucle adversarial debut-- num batch  3376
boucle adv

boucle adversarial fin
boucle adversarial debut-- num batch  3487
boucle adversarial fin
boucle adversarial debut-- num batch  3488
boucle adversarial fin
boucle adversarial debut-- num batch  3489
boucle adversarial fin
boucle adversarial debut-- num batch  3490
boucle adversarial fin
boucle adversarial debut-- num batch  3491
boucle adversarial fin
boucle adversarial debut-- num batch  3492
boucle adversarial fin
boucle adversarial debut-- num batch  3493
boucle adversarial fin
boucle adversarial debut-- num batch  3494
boucle adversarial fin
boucle adversarial debut-- num batch  3495
boucle adversarial fin
boucle adversarial debut-- num batch  3496
boucle adversarial fin
boucle adversarial debut-- num batch  3497
boucle adversarial fin
boucle adversarial debut-- num batch  3498
boucle adversarial fin
boucle adversarial debut-- num batch  3499
boucle adversarial fin
boucle adversarial debut-- num batch  3500
boucle adversarial fin
boucle adversarial debut-- num batch  3501
boucle adv

boucle adversarial fin
boucle adversarial debut-- num batch  3612
boucle adversarial fin
boucle adversarial debut-- num batch  3613
boucle adversarial fin
boucle adversarial debut-- num batch  3614
boucle adversarial fin
boucle adversarial debut-- num batch  3615
boucle adversarial fin
boucle adversarial debut-- num batch  3616
boucle adversarial fin
boucle adversarial debut-- num batch  3617
boucle adversarial fin
boucle adversarial debut-- num batch  3618
boucle adversarial fin
boucle adversarial debut-- num batch  3619
boucle adversarial fin
boucle adversarial debut-- num batch  3620
boucle adversarial fin
boucle adversarial debut-- num batch  3621
boucle adversarial fin
boucle adversarial debut-- num batch  3622
boucle adversarial fin
boucle adversarial debut-- num batch  3623
boucle adversarial fin
boucle adversarial debut-- num batch  3624
boucle adversarial fin
boucle adversarial debut-- num batch  3625
boucle adversarial fin
boucle adversarial debut-- num batch  3626
boucle adv

boucle adversarial fin
boucle adversarial debut-- num batch  3737
boucle adversarial fin
boucle adversarial debut-- num batch  3738
boucle adversarial fin
boucle adversarial debut-- num batch  3739
boucle adversarial fin
boucle adversarial debut-- num batch  3740
boucle adversarial fin
boucle adversarial debut-- num batch  3741
boucle adversarial fin
boucle adversarial debut-- num batch  3742
boucle adversarial fin
boucle adversarial debut-- num batch  3743
boucle adversarial fin
boucle adversarial debut-- num batch  3744
boucle adversarial fin
boucle adversarial debut-- num batch  3745
boucle adversarial fin
boucle adversarial debut-- num batch  3746
boucle adversarial fin
boucle adversarial debut-- num batch  3747
boucle adversarial fin
boucle adversarial debut-- num batch  3748
boucle adversarial fin
boucle adversarial debut-- num batch  3749
boucle adversarial fin
boucle adversarial debut-- num batch  3750
boucle adversarial fin
boucle adversarial debut-- num batch  3751
boucle adv

boucle adversarial fin
boucle adversarial debut-- num batch  3862
boucle adversarial fin
boucle adversarial debut-- num batch  3863
boucle adversarial fin
boucle adversarial debut-- num batch  3864
boucle adversarial fin
boucle adversarial debut-- num batch  3865
boucle adversarial fin
boucle adversarial debut-- num batch  3866
boucle adversarial fin
boucle adversarial debut-- num batch  3867
boucle adversarial fin
boucle adversarial debut-- num batch  3868
boucle adversarial fin
boucle adversarial debut-- num batch  3869
boucle adversarial fin
boucle adversarial debut-- num batch  3870
boucle adversarial fin
boucle adversarial debut-- num batch  3871
boucle adversarial fin
boucle adversarial debut-- num batch  3872
boucle adversarial fin
boucle adversarial debut-- num batch  3873
boucle adversarial fin
boucle adversarial debut-- num batch  3874
boucle adversarial fin
boucle adversarial debut-- num batch  3875
boucle adversarial fin
boucle adversarial debut-- num batch  3876
boucle adv

In [30]:
# [precision, recall, f_score, NDCG] = utility.test_model_all(Recom, test_df, train_df)

# utility.test_model_per_user_type(Recom, test_df, train_df, user_type_list, key_type)
# auc = utility.auc_per_user(Recom, test_df, train_df)
# print("AUC global is: ", auc)

precision_1	[0.2250000],	||	 precision_5	[0.1903642],	||	 precision_10	[0.1738079],	||	 precision_15	[0.1609603]
recall_1   	[0.0079383],	||	 recall_5   	[0.0339846],	||	 recall_10   	[0.0616546],	||	 recall_15   	[0.0856548]
f_measure_1	[0.0153355],	||	 f_measure_5	[0.0576732],	||	 f_measure_10	[0.0910214],	||	 f_measure_15	[0.1118100]
ndcg_1     	[0.2250000],	||	 ndcg_5     	[0.1972000],	||	 ndcg_10     	[0.1873616],	||	 ndcg_15     	[0.1825655]
Metrics for user type	 M
precision_1	[0.2641422],	||	 precision_5	[0.2106673],	||	 precision_10	[0.1901178],	||	 precision_15	[0.1753406]
recall_1	[0.0093639],	||	 recall_5	[0.0367411],	||	 recall_10	[0.0648360],	||	 recall_15	[0.0896665]
ndcg_1	[0.2641422],	||	 ndcg_5	[0.2213983],	||	 ndcg_10	[0.2074611],	||	 ndcg_15	[0.2006979]
AUC per user type	[0.8342900]
Metrics for user type	 F
precision_1	[0.1258046],	||	 precision_5	[0.1389116],	||	 precision_10	[0.1324751],	||	 precision_15	[0.1245173]
recall_1	[0.0043255],	||	 recall_5	[0.0269990],	

In [31]:
import matplotlib.pyplot as plt
import pandas as pd
from colour import Color

def savepdf_barplot_color_gradient(ymin = 0.5, ymax = 0.7, whis = 5, start_color='pink',end_color='blue',num_color=5, title='',axis_x = None, xlabel = '', axis_y1 = None, ylabel ='', plot_file = ''):
    
    fig = plt.figure()
    gs = fig.add_gridspec(1, 2, hspace=0, wspace=0)
    (ax1, ax2) = gs.subplots(sharex='col', sharey='row')
    
    red = Color(start_color)
    colors = list(red.range_to(Color(end_color), num_color))
    colors = [color.rgb for color in colors]
    
    X_axis = np.arange(len(axis_x))
    ax1.bar(X_axis, axis_y1, color=colors)
    ax1.hlines(y=axis_y1[0], xmin = 0, xmax = len(axis_x)-1, colors='black', linestyles='--', lw=1)
    
    plt.sca(ax1)
    plt.xticks(X_axis, axis_x, rotation =50)
    #plt.xlabel(xlabel)
    #fig.suptitle(title)
    plt.ylabel(ylabel, fontsize=18)
    plt.rcParams.update({'font.size': 13}) 
    plt.grid()
    
    plt.sca(ax2)
    ax2.boxplot(axis_y1, whis = whis)
    ax1.set_ylim(ymin, ymax)
    
    plt.tight_layout()
    plt.savefig(plot_file)

def savepdf_barplot_color_gradient2(start_color='pink',end_color='blue',num_color=20, title='',axis_x = None, xlabel = '', axis_y1 = None, ylabel ='', plot_file = ''):
    
    red = Color(start_color)
    colors = list(red.range_to(Color(end_color), num_color))
    colors = [color.rgb for color in colors]
    
    X_axis = np.arange(len(axis_x))
    plt.bar(X_axis, axis_y1, color=colors)
    
    
    plt.xticks(X_axis, axis_x, rotation =70)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid()
    
   # plt.tight_layout()
    plt.savefig(plot_file)